In [ ]:
"""
Plant Counting Research Pipeline - Publication-Grade Logging
All outputs saved to: ./experiments/yolov12x_<timestamp>/

Folder structure:
  yolov12x_<timestamp>/
  ├── config.yaml
  ├── env/
  │   ├── nvidia_smi.txt
  │   ├── pip_freeze.txt
  │   └── system.json
  ├── weights/
  │   ├── best.pt
  │   └── last.pt
  ├── training_curves/
  ├── detection_metrics/
  ├── counting_metrics/
  ├── visualizations/
  └── yolo_runs/
      ├── train/     (raw YOLO output: results.csv, plots, weights)
      └── val_test/  (raw YOLO val output)
"""

import os
import sys
import json
import time
import shutil
import random
import subprocess
from pathlib import Path
from datetime import datetime
from collections import defaultdict

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run(cmd: str) -> str:
    print(f"\n[CMD] {cmd}")
    out = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT)
    print(out)
    return out

print("=" * 60)
print("  INSTALLING DEPENDENCIES")
print("=" * 60)

run(
    "pip install -q "
    "git+https://github.com/sunsmarterjie/yolov12.git "
    "roboflow matplotlib scipy pandas pyyaml opencv-python-headless"
)

import yaml
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from ultralytics import YOLO
from roboflow import Roboflow

HOME = os.getcwd()
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n[INFO] Working directory: {HOME}")
print(f"[INFO] Timestamp: {TIMESTAMP}")
print(f"[INFO] Python: {sys.version}")
print(f"[INFO] PyTorch: {torch.__version__}")
print(f"[INFO] CUDA available: {torch.cuda.is_available()}")
print(f"[INFO] GPU count: {torch.cuda.device_count()}")

GPU_INFO = []
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    GPU_INFO.append({"id": i, "name": name, "memory_gb": round(mem, 1)})
    print(f"[INFO] GPU {i}: {name} ({mem:.1f} GB)")

# -----------------------------
# SECTION 1: EXP DIRS + ENV LOGS
# -----------------------------
MODEL_NAME = "yolov12x"
EXP_DIR = Path(HOME) / "experiments" / f"{MODEL_NAME}_{TIMESTAMP}"
EXP_DIR.mkdir(parents=True, exist_ok=True)

ENV_DIR       = EXP_DIR / "env"
WEIGHTS_DIR   = EXP_DIR / "weights"
TRAIN_CURVES  = EXP_DIR / "training_curves"
DET_METRICS   = EXP_DIR / "detection_metrics"
COUNT_METRICS = EXP_DIR / "counting_metrics"
VIS_DIR       = EXP_DIR / "visualizations"
YOLO_RUNS_DIR = EXP_DIR / "yolo_runs"

for d in [ENV_DIR, WEIGHTS_DIR, TRAIN_CURVES, DET_METRICS, COUNT_METRICS, VIS_DIR, YOLO_RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

try:
    nvsmi = run("nvidia-smi")
    (ENV_DIR / "nvidia_smi.txt").write_text(nvsmi, encoding="utf-8")
except Exception as e:
    (ENV_DIR / "nvidia_smi.txt").write_text(f"Failed to run nvidia-smi: {e}\n", encoding="utf-8")

try:
    pipfreeze = run(f"{sys.executable} -m pip freeze")
    (ENV_DIR / "pip_freeze.txt").write_text(pipfreeze, encoding="utf-8")
except Exception as e:
    (ENV_DIR / "pip_freeze.txt").write_text(f"Failed to run pip freeze: {e}\n", encoding="utf-8")

system_info = {
    "cwd": HOME,
    "timestamp": TIMESTAMP,
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_count": int(torch.cuda.device_count()),
    "gpus": GPU_INFO,
}
(ENV_DIR / "system.json").write_text(json.dumps(system_info, indent=2), encoding="utf-8")

print(f"\n[INFO] Experiment directory: {EXP_DIR}")

# -----------------------------
# SECTION 2: CONFIG (FABA BEANS)
# -----------------------------
# Use env var, do not hardcode secrets
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "gEPxiHcUHUH6siCWVXc6").strip()
WORKSPACE = "pjjagwe"
PROJECT   = "plant_weed_higher_resolution"
VERSION   = 28

# IMPORTANT: Use YOLO format for Ultralytics training
# If Roboflow supports "yolov12" for this dataset you can set that.
# Safe default is "yolov8" which produces YOLO labels + data.yaml.
DATASET_FORMAT = "yolov8"

if not ROBOFLOW_API_KEY:
    raise RuntimeError(
        "ROBOFLOW_API_KEY is not set.\n"
        "Set it and rerun:\n"
        "  export ROBOFLOW_API_KEY='YOUR_KEY'"
    )

# Training
MODEL_SIZE   = f"{MODEL_NAME}.pt"
IMGSZ        = 1280
EPOCHS       = 200
PATIENCE     = 25          # early stopping enabled
BATCH        = 32
DEVICE       = "0,1,2,3"
MULTI_SCALE  = False
WORKERS      = 8

OPTIMIZER    = "AdamW"
LR0          = 0.001
LRF          = 0.01
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 5.0

# Augmentation
CLOSE_MOSAIC = 15
MOSAIC       = 1.0
MIXUP        = 0.15
COPY_PASTE   = 0.15
FLIPLR       = 0.5
FLIPUD       = 0.5
TRANSLATE    = 0.05
SCALE        = 0.15
DEGREES      = 5.0
HSV_H        = 0.02
HSV_S        = 0.4
HSV_V        = 0.1

# Loss weights
BOX_LOSS = 0.15
CLS_LOSS = 0.5
DFL_LOSS = 1.5

# Detection eval thresholds (reporting)
VAL_CONF    = 0.001
VAL_NMS_IOU = 0.7

# Counting operating point + matching
CONF_THRESHOLD = 0.25
MATCH_IOU      = 0.2
PLANT_CLASS_ID = 0

MAX_VIZ_PER_SPLIT = 200

CONFIG = {
    "model": MODEL_NAME,
    "model_weights": MODEL_SIZE,
    "timestamp": TIMESTAMP,
    "train": {
        "imgsz": IMGSZ,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "batch": BATCH,
        "device": DEVICE,
        "multi_scale": MULTI_SCALE,
        "workers": WORKERS,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "lrf": LRF,
        "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": WARMUP_EPOCHS,
        "loss_weights": {"box": BOX_LOSS, "cls": CLS_LOSS, "dfl": DFL_LOSS},
        "augmentation": {
            "close_mosaic": CLOSE_MOSAIC,
            "mosaic": MOSAIC,
            "mixup": MIXUP,
            "copy_paste": COPY_PASTE,
            "fliplr": FLIPLR,
            "flipud": FLIPUD,
            "translate": TRANSLATE,
            "scale": SCALE,
            "degrees": DEGREES,
            "hsv_h": HSV_H,
            "hsv_s": HSV_S,
            "hsv_v": HSV_V,
        },
    },
    "eval": {
        "val_conf": VAL_CONF,
        "val_nms_iou": VAL_NMS_IOU,
        "counting_conf": CONF_THRESHOLD,
        "match_iou": MATCH_IOU,
        "plant_class_id": PLANT_CLASS_ID,
    },
    "dataset": {
        "workspace": WORKSPACE,
        "project": PROJECT,
        "version": VERSION,
        "format": DATASET_FORMAT,
    },
    "hardware": {
        "gpu_count": torch.cuda.device_count(),
        "gpus": GPU_INFO,
        "pytorch": torch.__version__,
        "python": sys.version.split()[0],
    },
}
with open(EXP_DIR / "config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(CONFIG, f, default_flow_style=False, sort_keys=False)

# -----------------------------
# SECTION 3: DOWNLOAD DATASET
# -----------------------------
print("\n" + "=" * 60)
print("  DOWNLOADING DATASET")
print("=" * 60)

t0 = time.time()
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download(DATASET_FORMAT)
print(f"[INFO] Dataset downloaded in {time.time() - t0:.1f}s")

DATA_ROOT = Path(dataset.location).resolve()

# Read class names from Roboflow-generated data.yaml if present
rf_data_yaml = DATA_ROOT / "data.yaml"
CLASS_NAMES = None
NUM_CLASSES = None
if rf_data_yaml.exists():
    try:
        rf_yaml = yaml.safe_load(rf_data_yaml.read_text(encoding="utf-8"))
        if isinstance(rf_yaml, dict) and "names" in rf_yaml and "nc" in rf_yaml:
            CLASS_NAMES = rf_yaml["names"]
            NUM_CLASSES = int(rf_yaml["nc"])
    except Exception:
        pass

if CLASS_NAMES is None or NUM_CLASSES is None:
    raise RuntimeError(
        f"Could not read class names from {rf_data_yaml}. "
        "Roboflow export may not be YOLO-style. Use a YOLO export format such as 'yolov8'."
    )

print(f"[INFO] Classes (nc={NUM_CLASSES}): {CLASS_NAMES}")

# Write absolute-path data yaml for stability
data_yaml = DATA_ROOT / "data_abs.yaml"
train_images = (DATA_ROOT / "train" / "images").resolve()
val_images   = (DATA_ROOT / "valid" / "images").resolve()
test_images  = (DATA_ROOT / "test" / "images").resolve()

data_yaml.write_text(
    f"nc: {NUM_CLASSES}\n"
    f"names: {CLASS_NAMES}\n"
    f"train: {str(train_images)}\n"
    f"val: {str(val_images)}\n"
    f"test: {str(test_images)}\n",
    encoding="utf-8",
)

print(f"[INFO] data_abs.yaml:\n{data_yaml.read_text(encoding='utf-8')}")

for split in ["train", "valid", "test"]:
    img_dir = DATA_ROOT / split / "images"
    count = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    print(f"[INFO] {split}: {count} images")

# -----------------------------
# SECTION 4: TRAIN
# -----------------------------
print("\n" + "=" * 60)
print(f"  TRAINING {MODEL_NAME.upper()} on {DEVICE} (Ultralytics DDP)")
print("=" * 60)

model = YOLO(MODEL_SIZE)
train_start = time.time()

results = model.train(
    data=str(data_yaml),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    amp=True,
    verbose=True,
    multi_scale=MULTI_SCALE,
    close_mosaic=CLOSE_MOSAIC,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    translate=TRANSLATE,
    scale=SCALE,
    degrees=DEGREES,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    box=BOX_LOSS,
    cls=CLS_LOSS,
    dfl=DFL_LOSS,
    plots=True,
    save=True,
    save_period=10,
    project=str(YOLO_RUNS_DIR),
    name="train",
    exist_ok=False,
)

train_time = time.time() - train_start
print(f"\n[INFO] Training completed in {train_time / 60:.1f} min ({train_time / 3600:.2f} hr)")

CONFIG["training_time_minutes"] = round(train_time / 60, 2)
with open(EXP_DIR / "config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(CONFIG, f, default_flow_style=False, sort_keys=False)

YOLO_TRAIN_DIR = Path(model.trainer.save_dir).resolve()
print(f"[INFO] YOLO train dir: {YOLO_TRAIN_DIR}")

for wt in ["best.pt", "last.pt"]:
    src = YOLO_TRAIN_DIR / "weights" / wt
    if src.exists():
        shutil.copy2(src, WEIGHTS_DIR / wt)
        print(f"[INFO] Saved {wt} ({src.stat().st_size / 1024 / 1024:.1f} MB)")

# -----------------------------
# SECTION 5: TRAINING CURVES
# -----------------------------
print("\n" + "=" * 60)
print("  GENERATING TRAINING CURVES")
print("=" * 60)

results_csv = YOLO_TRAIN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    curve_defs = {
        "train/box_loss": ("Train Box Loss", "train_box_loss.png"),
        "train/cls_loss": ("Train Cls Loss", "train_cls_loss.png"),
        "train/dfl_loss": ("Train DFL Loss", "train_dfl_loss.png"),
        "val/box_loss":   ("Val Box Loss",   "val_box_loss.png"),
        "val/cls_loss":   ("Val Cls Loss",   "val_cls_loss.png"),
        "val/dfl_loss":   ("Val DFL Loss",   "val_dfl_loss.png"),
        "lr/pg0":         ("Learning Rate",  "lr_schedule.png"),
        "metrics/mAP50(B)":        ("mAP@0.5",      "mAP50.png"),
        "metrics/mAP50-95(B)":     ("mAP@0.5:0.95", "mAP50-95.png"),
        "metrics/precision(B)":    ("Precision",   "precision.png"),
        "metrics/recall(B)":       ("Recall",      "recall.png"),
    }

    for col, (title, fname) in curve_defs.items():
        if col in df.columns:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(df["epoch"], df[col], linewidth=2)
            ax.set_xlabel("Epoch", fontsize=14)
            ax.set_ylabel(title, fontsize=14)
            ax.set_title(f"{MODEL_NAME.upper()} - {title}", fontsize=16, fontweight="bold")
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            fig.savefig(TRAIN_CURVES / fname, dpi=300, bbox_inches="tight")
            plt.close(fig)
            print(f"  [PLOT] {fname}")

    fig, axes = plt.subplots(3, 4, figsize=(28, 18))
    fig.suptitle(f"{MODEL_NAME.upper()} Training Curves", fontsize=20, fontweight="bold", y=0.98)
    plot_order = list(curve_defs.keys())
    for idx, col in enumerate(plot_order):
        row, c = divmod(idx, 4)
        ax = axes[row][c]
        if col in df.columns:
            ax.plot(df["epoch"], df[col], linewidth=1.5)
            ax.set_title(curve_defs[col][0], fontsize=13, fontweight="bold")
            ax.set_xlabel("Epoch", fontsize=11)
            ax.grid(True, alpha=0.3)
    axes[2][3].axis("off")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(TRAIN_CURVES / "all_curves_combined.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("  [PLOT] all_curves_combined.png")
else:
    print("[WARNING] results.csv not found in train output")

# -----------------------------
# SECTION 6: DETECTION METRICS (VAL ON TEST)
# -----------------------------
print("\n" + "=" * 60)
print("  DETECTION METRICS - YOLO VALIDATION (test split)")
print("=" * 60)

best_path = WEIGHTS_DIR / "best.pt"
if not best_path.exists():
    raise FileNotFoundError(f"best.pt not found at: {best_path}")

model_best = YOLO(str(best_path))

metrics = model_best.val(
    split="test",
    data=str(data_yaml),
    imgsz=IMGSZ,
    conf=VAL_CONF,
    iou=VAL_NMS_IOU,
    verbose=True,
    plots=True,
    project=str(YOLO_RUNS_DIR),
    name="val_test",
    exist_ok=True,
)

YOLO_VAL_DIR = (YOLO_RUNS_DIR / "val_test").resolve()
print(f"[INFO] YOLO val dir: {YOLO_VAL_DIR}")

if YOLO_VAL_DIR.exists():
    for f in YOLO_VAL_DIR.glob("*.png"):
        shutil.copy2(f, DET_METRICS / f.name)
        print(f"  [COPY] {f.name}")

det_summary = {
    "mAP50": round(float(metrics.box.map50), 4) if hasattr(metrics, "box") and hasattr(metrics.box, "map50") else None,
    "mAP50-95": round(float(metrics.box.map), 4) if hasattr(metrics, "box") and hasattr(metrics.box, "map") else None,
    "precision": round(float(metrics.box.mp), 4) if hasattr(metrics, "box") and hasattr(metrics.box, "mp") else None,
    "recall": round(float(metrics.box.mr), 4) if hasattr(metrics, "box") and hasattr(metrics.box, "mr") else None,
}
with open(DET_METRICS / "detection_summary.yaml", "w", encoding="utf-8") as f:
    yaml.dump(det_summary, f)
print(f"\n[INFO] Detection summary: {det_summary}")

# -----------------------------
# SECTION 7: PER-IMAGE COUNTING + VISUALIZATION
# -----------------------------
print("\n" + "=" * 60)
print("  PER-IMAGE PLANT COUNTING + VISUALIZATION")
print("=" * 60)

SPLITS = {"train": DATA_ROOT / "train", "valid": DATA_ROOT / "valid", "test": DATA_ROOT / "test"}
GREEN, RED, ORANGE = (0, 255, 0), (0, 0, 255), (0, 165, 255)

def compute_iou(a, b):
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(xi2 - xi1, 0) * max(yi2 - yi1, 0)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls, cx, cy, bw, bh = map(float, parts[:5])
            if int(cls) == PLANT_CLASS_ID:
                boxes.append([
                    (cx - bw / 2) * img_w,
                    (cy - bh / 2) * img_h,
                    (cx + bw / 2) * img_w,
                    (cy + bh / 2) * img_h,
                ])
    return boxes

def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = (2 * p * r) / (p + r) if (p + r) else 0.0
    return p, r, f

all_image_results = []
split_totals = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0, "gt": 0, "pred": 0})

for split_name, split_root in SPLITS.items():
    img_dir, lbl_dir = split_root / "images", split_root / "labels"
    out_dir = VIS_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    img_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"):
        img_paths.extend(list(img_dir.glob(ext)))
    img_paths = sorted(img_paths)

    print(f"\n{'='*60}\n  {split_name.upper()}: {len(img_paths)} images\n{'='*60}")

    viz_set = set(img_paths)
    if MAX_VIZ_PER_SPLIT is not None and len(img_paths) > int(MAX_VIZ_PER_SPLIT):
        viz_set = set(random.sample(img_paths, int(MAX_VIZ_PER_SPLIT)))

    split_start = time.time()
    for img_idx, img_path in enumerate(img_paths, 1):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        gt_boxes = load_gt_boxes(lbl_dir / f"{img_path.stem}.txt", w, h)
        split_totals[split_name]["gt"] += len(gt_boxes)

        res = model_best.predict(
            img_path,
            conf=CONF_THRESHOLD,
            iou=VAL_NMS_IOU,
            imgsz=IMGSZ,
            verbose=False,
        )[0]

        pred_boxes, pred_confs = [], []
        for box, cls, conf in zip(
            res.boxes.xyxy.cpu().numpy(),
            res.boxes.cls.cpu().numpy(),
            res.boxes.conf.cpu().numpy(),
        ):
            if int(cls) == PLANT_CLASS_ID:
                pred_boxes.append(box.tolist())
                pred_confs.append(float(conf))
        split_totals[split_name]["pred"] += len(pred_boxes)

        matched_gt, matched_pred = set(), set()
        for p_idx, pb in enumerate(pred_boxes):
            best_iou, best_g = 0.0, -1
            for g_idx, gb in enumerate(gt_boxes):
                if g_idx in matched_gt:
                    continue
                s = compute_iou(pb, gb)
                if s > best_iou:
                    best_iou, best_g = s, g_idx
            if best_iou >= MATCH_IOU and best_g >= 0:
                matched_gt.add(best_g)
                matched_pred.add(p_idx)

        tp = len(matched_gt)
        fp = len(pred_boxes) - len(matched_pred)
        fn = len(gt_boxes) - len(matched_gt)

        split_totals[split_name]["tp"] += tp
        split_totals[split_name]["fp"] += fp
        split_totals[split_name]["fn"] += fn

        avg_conf = float(np.mean(pred_confs)) if pred_confs else 0.0

        all_image_results.append({
            "split": split_name,
            "image": img_path.name,
            "gt_count": len(gt_boxes),
            "pred_count": len(pred_boxes),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "count_error": len(pred_boxes) - len(gt_boxes),
            "abs_count_error": abs(len(pred_boxes) - len(gt_boxes)),
            "avg_confidence": round(avg_conf, 4),
        })

        if img_path in viz_set:
            for g_idx, gb in enumerate(gt_boxes):
                c = GREEN if g_idx in matched_gt else ORANGE
                cv2.rectangle(img, (int(gb[0]), int(gb[1])), (int(gb[2]), int(gb[3])), c, 2)
            for p_idx, pb in enumerate(pred_boxes):
                if p_idx not in matched_pred:
                    cv2.rectangle(img, (int(pb[0]), int(pb[1])), (int(pb[2]), int(pb[3])), RED, 2)

            cv2.putText(
                img,
                f"GT:{len(gt_boxes)} Pred:{len(pred_boxes)} TP:{tp} FP:{fp} FN:{fn}",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (255, 255, 255),
                3,
            )
            cv2.imwrite(str(out_dir / f"{img_path.stem}.png"), img)

        print(
            f"  [{img_idx:4d}/{len(img_paths)}] {img_path.name:40s} "
            f"GT:{len(gt_boxes):3d} Pred:{len(pred_boxes):3d} TP:{tp:3d} FP:{fp:3d} FN:{fn:3d} Err:{len(pred_boxes)-len(gt_boxes):+4d}"
        )

    elapsed = time.time() - split_start
    per_img = elapsed / max(1, len(img_paths))
    print(f"\n  [INFO] {split_name} done in {elapsed:.1f}s ({per_img:.2f}s/img)")
    if MAX_VIZ_PER_SPLIT is not None:
        print(f"  [INFO] Saved annotated images: {min(len(img_paths), int(MAX_VIZ_PER_SPLIT))}")

# -----------------------------
# SECTION 8: COUNTING METRICS & PLOTS
# -----------------------------
print("\n" + "=" * 60)
print("  COUNTING METRICS & RESEARCH PLOTS")
print("=" * 60)

df_results = pd.DataFrame(all_image_results)
df_results.to_csv(COUNT_METRICS / "per_image_results.csv", index=False)

split_rows = []
for sn in ["train", "valid", "test"]:
    d = split_totals[sn]
    p_, r_, f_ = prf(d["tp"], d["fp"], d["fn"])

    sdf = df_results[df_results["split"] == sn]
    mae_  = float(sdf["abs_count_error"].mean()) if len(sdf) else 0.0
    rmse_ = float(np.sqrt((sdf["count_error"].astype(float) ** 2).mean())) if len(sdf) else 0.0
    mape_ = float((sdf["abs_count_error"] / sdf["gt_count"].clip(lower=1) * 100).mean()) if len(sdf) else 0.0

    if len(sdf) > 1 and float(sdf["gt_count"].std()) > 0:
        r2_ = float(stats.pearsonr(sdf["gt_count"], sdf["pred_count"])[0] ** 2)
    else:
        r2_ = None

    split_rows.append({
        "split": sn,
        "images": int(len(sdf)),
        "gt_plants": int(d["gt"]),
        "pred_plants": int(d["pred"]),
        "TP": int(d["tp"]),
        "FP": int(d["fp"]),
        "FN": int(d["fn"]),
        "precision": round(p_, 4),
        "recall": round(r_, 4),
        "f1": round(f_, 4),
        "MAE": round(mae_, 2),
        "RMSE": round(rmse_, 2),
        "MAPE(%)": round(mape_, 2),
        "R2": round(r2_, 4) if r2_ is not None else "N/A",
    })

    print(f"  [{sn:5s}] P:{p_:.3f} R:{r_:.3f} F1:{f_:.3f} MAE:{mae_:.2f} RMSE:{rmse_:.2f}" + (f" R2:{r2_:.4f}" if r2_ is not None else ""))

overall_tp = sum(split_totals[s]["tp"] for s in ["train", "valid", "test"])
overall_fp = sum(split_totals[s]["fp"] for s in ["train", "valid", "test"])
overall_fn = sum(split_totals[s]["fn"] for s in ["train", "valid", "test"])
overall_gt = sum(split_totals[s]["gt"] for s in ["train", "valid", "test"])
overall_pr = sum(split_totals[s]["pred"] for s in ["train", "valid", "test"])
p_o, r_o, f_o = prf(overall_tp, overall_fp, overall_fn)

mae_o  = float(df_results["abs_count_error"].mean()) if len(df_results) else 0.0
rmse_o = float(np.sqrt((df_results["count_error"].astype(float) ** 2).mean())) if len(df_results) else 0.0
mape_o = float((df_results["abs_count_error"] / df_results["gt_count"].clip(lower=1) * 100).mean()) if len(df_results) else 0.0
r2_o   = float(stats.pearsonr(df_results["gt_count"], df_results["pred_count"])[0] ** 2) if len(df_results) > 1 and float(df_results["gt_count"].std()) > 0 else None

split_rows.append({
    "split": "OVERALL",
    "images": int(len(df_results)),
    "gt_plants": int(overall_gt),
    "pred_plants": int(overall_pr),
    "TP": int(overall_tp),
    "FP": int(overall_fp),
    "FN": int(overall_fn),
    "precision": round(p_o, 4),
    "recall": round(r_o, 4),
    "f1": round(f_o, 4),
    "MAE": round(mae_o, 2),
    "RMSE": round(rmse_o, 2),
    "MAPE(%)": round(mape_o, 2),
    "R2": round(r2_o, 4) if r2_o is not None else "N/A",
})

df_summary = pd.DataFrame(split_rows)
df_summary.to_csv(COUNT_METRICS / "per_split_summary.csv", index=False)

print(f"\n  [OVERALL] P:{p_o:.3f} R:{r_o:.3f} F1:{f_o:.3f} MAE:{mae_o:.2f} RMSE:{rmse_o:.2f}" + (f" R2:{r2_o:.4f}" if r2_o is not None else ""))

# Plot 1: GT vs Pred scatter
gt_c = df_results["gt_count"].values.astype(float)
pr_c = df_results["pred_count"].values.astype(float)

fig, ax = plt.subplots(figsize=(10, 10))
for sp, marker in [("train", "o"), ("valid", "s"), ("test", "^")]:
    m = (df_results["split"] == sp).values
    ax.scatter(gt_c[m], pr_c[m], alpha=0.6, s=50, marker=marker, edgecolors="white", linewidth=0.5, label=sp)

mx = max(float(gt_c.max()) if len(gt_c) else 0.0, float(pr_c.max()) if len(pr_c) else 0.0) * 1.1 + 1e-6
ax.plot([0, mx], [0, mx], "k--", linewidth=1.5, label="Perfect (y=x)")

if len(df_results) > 1:
    slope, intercept, _, _, _ = stats.linregress(gt_c, pr_c)
    xf = np.linspace(0, mx, 100)
    ax.plot(xf, slope * xf + intercept, "r-", linewidth=2, label=f"Fit: y={slope:.3f}x+{intercept:.2f}")

ax.set_xlabel("Ground Truth Count", fontsize=14)
ax.set_ylabel("Predicted Count", fontsize=14)
title_r2 = f"{r2_o:.4f}" if r2_o is not None else "N/A"
ax.set_title(f"{MODEL_NAME.upper()} - GT vs Predicted\nR2={title_r2} | MAE={mae_o:.2f} | RMSE={rmse_o:.2f}", fontsize=15, fontweight="bold")
ax.legend(fontsize=12)
ax.set_xlim(0, mx)
ax.set_ylim(0, mx)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(COUNT_METRICS / "gt_vs_pred_scatter.png", dpi=300)
plt.close(fig)
print("[PLOT] gt_vs_pred_scatter.png")

# Plot 2: Error histogram
fig, ax = plt.subplots(figsize=(12, 7))
errs = df_results["count_error"].values.astype(float)
bins = max(30, len(set(errs.tolist()))) if len(errs) else 30
ax.hist(errs, bins=bins, edgecolor="white", alpha=0.8)
ax.axvline(0, color="red", linestyle="--", linewidth=2, label="Zero error")
ax.axvline(float(errs.mean()) if len(errs) else 0.0, color="orange", linestyle="-", linewidth=2, label=f"Mean: {float(errs.mean()) if len(errs) else 0.0:.2f}")
ax.set_xlabel("Counting Error (Pred - GT)", fontsize=14)
ax.set_ylabel("Images", fontsize=14)
ax.set_title(f"{MODEL_NAME.upper()} - Error Distribution | MAE={np.abs(errs).mean() if len(errs) else 0.0:.2f}", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(COUNT_METRICS / "counting_error_histogram.png", dpi=300)
plt.close(fig)
print("[PLOT] counting_error_histogram.png")

# Plot 3: Bland-Altman
fig, ax = plt.subplots(figsize=(12, 7))
means = (gt_c + pr_c) / 2.0
diffs = pr_c - gt_c
md = float(diffs.mean()) if len(diffs) else 0.0
sd = float(diffs.std()) if len(diffs) else 0.0
ax.scatter(means, diffs, alpha=0.5, s=40, edgecolors="white", linewidth=0.5)
ax.axhline(md, color="red", linestyle="-", linewidth=2, label=f"Mean diff: {md:.2f}")
ax.axhline(md + 1.96 * sd, color="gray", linestyle="--", linewidth=1.5, label=f"+1.96 SD: {md + 1.96 * sd:.2f}")
ax.axhline(md - 1.96 * sd, color="gray", linestyle="--", linewidth=1.5, label=f"-1.96 SD: {md - 1.96 * sd:.2f}")
ax.set_xlabel("Mean of GT and Predicted", fontsize=14)
ax.set_ylabel("Difference (Pred - GT)", fontsize=14)
ax.set_title(f"{MODEL_NAME.upper()} - Bland-Altman (Agreement Analysis)", fontsize=15, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(COUNT_METRICS / "bland_altman_plot.png", dpi=300)
plt.close(fig)
print("[PLOT] bland_altman_plot.png")

# Plot 4: Per-split comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
bar_s = ["train", "valid", "test"]
x = np.arange(len(bar_s))

for i, metric in enumerate(["precision", "recall", "f1"]):
    v = [float(df_summary[df_summary["split"] == s][metric].values[0]) for s in bar_s]
    axes[0].bar(x + i * 0.25, v, 0.25, label=metric.upper(), alpha=0.85)
axes[0].set_xticks(x + 0.25)
axes[0].set_xticklabels([s.capitalize() for s in bar_s])
axes[0].set_title("P / R / F1", fontsize=14, fontweight="bold")
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis="y", alpha=0.3)

for i, metric in enumerate(["MAE", "RMSE"]):
    v = [float(df_summary[df_summary["split"] == s][metric].values[0]) for s in bar_s]
    axes[1].bar(x + i * 0.35, v, 0.35, label=metric, alpha=0.85)
axes[1].set_xticks(x + 0.175)
axes[1].set_xticklabels([s.capitalize() for s in bar_s])
axes[1].set_title("MAE / RMSE", fontsize=14, fontweight="bold")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

r2v = []
for s in bar_s:
    val = df_summary[df_summary["split"] == s]["R2"].values[0]
    r2v.append(float(val) if val != "N/A" else 0.0)
axes[2].bar(bar_s, r2v, alpha=0.85)
for i, v in enumerate(r2v):
    axes[2].text(i, v + 0.02, f"{v:.4f}", ha="center", fontsize=12, fontweight="bold")
axes[2].set_title("R2", fontsize=14, fontweight="bold")
axes[2].set_ylim(0, 1.1)
axes[2].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(COUNT_METRICS / "per_split_comparison.png", dpi=300)
plt.close(fig)
print("[PLOT] per_split_comparison.png")

# -----------------------------
# SECTION 9: FINAL SUMMARY + ZIP
# -----------------------------
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)

print(f"\n  Model:     {MODEL_NAME}")
print(f"  imgsz:     {IMGSZ}")
print(f"  Training:  {train_time/60:.1f} min on {DEVICE}")
print(f"  Early stopping: patience={PATIENCE}")
print(f"\n  DETECTION:  mAP50={det_summary.get('mAP50','?')}  mAP50-95={det_summary.get('mAP50-95','?')}")
print(f"  COUNTING:   P={p_o:.4f}  R={r_o:.4f}  F1={f_o:.4f}")
print(f"              MAE={mae_o:.2f}  RMSE={rmse_o:.2f}  MAPE={mape_o:.2f}%  R2={(f'{r2_o:.4f}' if r2_o is not None else 'N/A')}")
print(f"\n  Experiment: {EXP_DIR}")

zip_path = f"{HOME}/experiment_{MODEL_NAME}_{TIMESTAMP}"
shutil.make_archive(zip_path, "zip", EXP_DIR)
print(f"\n[INFO] Zip: {zip_path}.zip")

total_time = time.time() - train_start
print(f"[INFO] Total: {total_time/60:.1f} min")
print("\nDone!")
